# Sample Filtering, Winsorisation and the Rank Transformation

Final preparation of the estimation sample, following Drobetz and Otto (2021).

Three operations, in order:

1. **Sample filter** — retain security-months with a complete set of characteristics
   and a realised subsequent return, subject to a minimum breadth requirement;
2. **Winsorisation** at the 1st and 99th percentiles, cross-sectionally;
3. **Rank transformation** onto the interval $(-1,+1)$, cross-sectionally.

## Two versions of the sample

The source study requires complete information across all twenty-two characteristics,
which discards any security-month with a single missing value. Because missingness is
concentrated among smaller firms and among securities delisted earlier in the sample,
this restriction interacts with the survivorship-free construction of the universe: it
removes disproportionately the firms whose retention motivated that construction.

Both samples are therefore built here. The **complete-case** sample reproduces the
source study; the **imputed** sample replaces missing characteristics with the
contemporaneous cross-sectional median, following Gu, Kelly and Xiu (2020). The two
are carried forward in parallel, so that the effect of the treatment on predictive
performance can be measured rather than assumed.

Order matters. Imputation precedes filtering, since its purpose is to make otherwise
incomplete observations usable. Winsorisation and ranking follow the filter, so that
the percentiles and ranks are computed on the sample actually estimated.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Tesi/dati"

panel = pd.read_parquet(os.path.join(DATA_DIR, "panel_characteristics.parquet"))
panel = panel.set_index(["symbol", "date"]).sort_index()

print(f"Panel: {len(panel):,} security-months, "
      f"{panel.index.get_level_values('symbol').nunique():,} securities")
print(f"       {panel.index.get_level_values('date').min():%Y-%m} to "
      f"{panel.index.get_level_values('date').max():%Y-%m}")

## 2. Functions

In [ ]:
"""Sample filtering, winsorisation and the rank transformation."""
import numpy as np
import pandas as pd

CHARS = ["size", "bm", "operatingprofitability", "totalassetgrowth", "ret_2_12",
         "issues_1_36", "accrualschange", "roa", "investment", "ret_1",
         "ret_12_36", "dy", "beta", "vola", "turnover", "debttoprice",
         "salestoprice", "cftoprice", "earningstoprice", "issues_1_12",
         "fcdispersion", "grossprofitability"]


def impute_cross_sectional_median(p, chars=CHARS):
    """
    Replace missing characteristics with the cross-sectional median of the
    month, following Gu, Kelly and Xiu (2020).

    The median is contemporaneous, so no information from later periods enters
    the imputation. Securities with no characteristic at all are left untouched
    and removed by the subsequent filter, since imputing every value would
    manufacture an observation rather than complete one.
    """
    p = p.copy()
    has_any = p[chars].notna().any(axis=1)
    for c in chars:
        med = p.groupby(level="date")[c].transform("median")
        p.loc[has_any, c] = p.loc[has_any, c].fillna(med[has_any])
    return p


def filter_sample(p, chars=CHARS, min_firms=50, min_mv=25.0, require_size=True):
    """
    Retain security-months with a complete set of characteristics and a
    realised subsequent return.

    Drobetz and Otto additionally require at least `min_firms` firms above
    `min_mv` million in market capitalisation in every month; this determines
    the start of their sample. Months failing the requirement are dropped.
    """
    ok = p[chars + ["target"]].notna().all(axis=1)
    d = p[ok].copy()

    if require_size:
        big = d["MV"] >= min_mv
        n_big = big.groupby(level="date").sum()
        valid = n_big[n_big >= min_firms].index
        dropped = sorted(set(d.index.get_level_values("date")) - set(valid))
        d = d[d.index.get_level_values("date").isin(valid)]
    else:
        dropped = []

    return d, dropped


def winsorise(p, cols, lower=0.01, upper=0.99):
    """
    Winsorise cross-sectionally, month by month, at the 1st and 99th
    percentiles, following Drobetz and Otto (2021).

    Applied within each cross-section rather than over the pooled sample, so
    that no information from future periods enters the transformation.
    """
    p = p.copy()
    for c in cols:
        g = p.groupby(level="date")[c]
        lo = g.transform(lambda s: s.quantile(lower))
        hi = g.transform(lambda s: s.quantile(upper))
        p[c] = p[c].clip(lo, hi)
    return p


def rank_transform(p, chars=CHARS):
    """
    Map each characteristic to its cross-sectional rank within the month,
    rescaled onto the interval (-1, +1).

    Follows Gu, Kelly and Xiu (2020) and Drobetz and Otto (2021). The
    transformation is computed one month at a time and is invariant to the
    units in which a characteristic is expressed, which removes the remaining
    differences of scale against the source study.
    """
    p = p.copy()
    for c in chars:
        r = p.groupby(level="date")[c].rank(method="average", pct=True)
        p[c] = 2.0 * r - 1.0
    return p


## 3. Sample period

The estimation sample begins in January 2000. Data were retrieved from 1996 so that
characteristics requiring up to thirty-six months of history are available from the
first month; those earlier months now fall away.

In [ ]:
SAMPLE_START = "2000-01-01"

panel = panel[panel.index.get_level_values("date") >= SAMPLE_START]
print(f"From {SAMPLE_START[:7]}: {len(panel):,} security-months, "
      f"{panel.index.get_level_values('symbol').nunique():,} securities")

print("\nMissingness by characteristic:")
miss = (100 * panel[CHARS].isna().mean()).sort_values(ascending=False)
for ch, m in miss.items():
    print(f"  {ch:24s} {m:>5.1f}%")

## 4. Complete-case sample

The breadth requirement of the source study — at least fifty firms above €25 million
in market capitalisation in every month — determines the start of their sample. It is
applied here for consistency, though the later starting date makes it unlikely to
bind.

In [ ]:
cc, dropped_cc = filter_sample(panel, min_firms=50, min_mv=25.0)

print(f"Complete case: {len(cc):,} security-months, "
      f"{cc.index.get_level_values('symbol').nunique():,} securities")
print(f"Months failing the breadth requirement: {len(dropped_cc)}")
if dropped_cc:
    print(f"  {dropped_cc[0]:%Y-%m} to {dropped_cc[-1]:%Y-%m}")

if len(cc):
    per_month_cc = cc.groupby(level="date").size()
    print(f"\nFirms per month: mean {per_month_cc.mean():.0f}, "
          f"min {per_month_cc.min():.0f}, max {per_month_cc.max():.0f}")
    print("(Drobetz and Otto report 832 for nineteen countries over 1990-2020)")
else:
    print("\nNo security-month has a complete set of characteristics.")

## 5. Imputed sample

Missing characteristics are replaced with the median of the same month, which uses no
information from later periods. Securities with no characteristic at all are left
untouched: imputing every value would manufacture an observation rather than complete
one, and such securities are removed by the filter.

In [ ]:
panel_imp = impute_cross_sectional_median(panel)
im, dropped_im = filter_sample(panel_imp, min_firms=50, min_mv=25.0)

print(f"Imputed: {len(im):,} security-months, "
      f"{im.index.get_level_values('symbol').nunique():,} securities")

per_month_im = im.groupby(level="date").size()
print(f"Firms per month: mean {per_month_im.mean():.0f}, "
      f"min {per_month_im.min():.0f}, max {per_month_im.max():.0f}")

if len(cc):
    n_cc = cc.index.get_level_values("symbol").nunique()
    n_im = im.index.get_level_values("symbol").nunique()
    print(f"\nRelative to complete case: {len(im)/len(cc):.2f}x observations, "
          f"{n_im/n_cc:.2f}x securities")

### What the complete-case restriction removes

Whether the two samples differ in composition, and not merely in size, is the
question that motivates carrying both forward.

In [ ]:
only_imp = im.index.difference(cc.index)
if len(only_imp) and len(cc):
    a, b = im.loc[only_imp], cc
    print(f"Observations recovered by imputation: {len(only_imp):,}")
    print(f"Securities in the recovered set    : "
          f"{only_imp.get_level_values('symbol').nunique():,}\n")
    print(f"{'':24s} {'recovered':>12s} {'complete case':>14s}")
    for lbl, col in [("median market value", "MV"), ("median beta", "beta"),
                     ("median volatility", "vola"), ("median turnover", "turnover")]:
        if col in a.columns:
            print(f"  {lbl:22s} {a[col].median():>12.3f} {b[col].median():>14.3f}")
    print("\nA lower median market value in the recovered set would indicate that the")
    print("complete-case restriction removes smaller firms selectively.")
else:
    print("Nothing to compare.")

## 6. Winsorisation

At the 1st and 99th percentiles, within each month. Drobetz and Otto apply this to
the characteristics and to excess returns alike, noting that this departs from Gu,
Kelly and Xiu, who instead use a Huber objective — a device unavailable to the
dimension-reduction methods and therefore incompatible with comparison across
models.

In [ ]:
TO_WINSORISE = CHARS + ["target"]

cc_w = winsorise(cc, TO_WINSORISE, lower=0.01, upper=0.99)
im_w = winsorise(im, TO_WINSORISE, lower=0.01, upper=0.99)

ref = cc if len(cc) else im
ref_w = cc_w if len(cc) else im_w
print(f"Effect on dispersion ({'complete-case' if len(cc) else 'imputed'} sample):")
print(f"{'':24s} {'before':>10s} {'after':>10s}")
for ch in ["target", "salestoprice", "debttoprice", "turnover", "beta"]:
    print(f"  {ch:22s} {ref[ch].std():>10.3f} {ref_w[ch].std():>10.3f}")

## 7. Rank transformation

Each characteristic is replaced by its cross-sectional rank within the month,
rescaled onto $(-1,+1)$. The transformation is invariant to the units in which a
characteristic is expressed, which removes the remaining differences of scale against
the source study, and bounds every predictor identically — a property the penalised
and tree-based methods both rely on.

In [ ]:
cc_r = rank_transform(cc_w)
im_r = rank_transform(im_w)

chk = cc_r if len(cc_r) else im_r
label = "Complete-case" if len(cc_r) else "Imputed"
print(f"{label} sample after ranking:")
st = chk[CHARS].describe().T[["min", "mean", "max"]]
print(f"  minimum across characteristics : {st['min'].min():.3f}")
print(f"  maximum across characteristics : {st['max'].max():.3f}")
print(f"  mean of means                  : {st['mean'].mean():.4f}")
print(f"  remaining missing values       : {int(chk[CHARS].isna().sum().sum())}")

print("\nDistribution of one characteristic, which should be uniform:")
print(chk["size"].describe(percentiles=[.25, .5, .75]).round(3).to_string())

### The dependent variable is not ranked

Only the predictors are transformed. The target remains the winsorised excess return,
since the models are estimated to forecast a return rather than a rank.

In [ ]:
chk = cc_r if len(cc_r) else im_r
label = "complete-case" if len(cc_r) else "imputed"
print(f"Target, {label} sample:")
print(chk["target"].describe(percentiles=[.01, .5, .99]).round(4).to_string())
print(f"\nMean monthly excess return: {100*chk['target'].mean():.3f}%")
print(f"Standard deviation        : {100*chk['target'].std():.2f}%")
print("(Drobetz and Otto report 0.51% and 4.55%)")

## 8. Country and sector indicators

Drobetz and Otto augment the characteristics with country indicators. Sector
membership is recorded in the universe file and is added here as well, following Gu,
Kelly and Xiu, who interact characteristics with industry dummies.

In [ ]:
master = pd.read_csv(os.path.join(DATA_DIR, "universo_master.csv"), dtype=str)
info = master.set_index("Symbol")[["country", "Sector"]]

for name, d in [("complete case", cc_r), ("imputed", im_r)]:
    if not len(d):
        print(f"{name}: empty"); continue
    d["country"] = d.index.get_level_values("symbol").map(info["country"])
    d["sector"] = d.index.get_level_values("symbol").map(info["Sector"])
    print(f"{name}: {d['country'].nunique()} countries, {d['sector'].nunique()} sectors")

chk = cc_r if len(cc_r) else im_r
print("\nObservations by country:")
print(chk["country"].value_counts().to_string())

## 9. Save

In [ ]:
KEEP = ["target", "retx", "MV", "country", "sector"] + CHARS

for name, d in [("estimation_complete_case", cc_r), ("estimation_imputed", im_r)]:
    cols = [c for c in KEEP if c in d.columns]
    out = d[cols].reset_index()
    p = os.path.join(DATA_DIR, f"{name}.parquet")
    out.to_parquet(p, index=False)
    print(f"  {name+'.parquet':36s} {len(out):>10,} rows x {out.shape[1]} cols  "
          f"({os.path.getsize(p)/1e6:.1f} MB)")

print("\nBoth samples are ready for estimation. The complete-case sample reproduces")
print("the source study; the imputed sample is carried forward for comparison.")